# Diagnóstico 06 — Carregamento integrado do Dashboard ENEM

## Objetivo

Verificar se a redução de memória observada no Diagnóstico 05, ao particionar as bases `grupo` e `dsp` por `TIPO_REFERENCIA_TERRITORIAL`, permanece quando as quatro bases são carregadas no mesmo processo.

O teste reproduz a sequência implementada em `features_names.py`:

1. leitura de `painel`, `faixa`, `grupo` e `dsp`;
2. leitura do shapefile;
3. criação do mapa de códigos e nomes municipais;
4. aplicação de `_preparar_tabela()` às quatro bases;
5. retorno das tabelas preparadas.

A lógica de harmonização territorial será importada do próprio módulo do projeto. Não será criada uma implementação paralela dessa etapa.

## Cenários

- **Original:** as quatro bases são lidas dos Parquets originais.
- **Particionado:** `painel` e `faixa` continuam originais; `grupo` e `dsp` são lidas das variantes particionadas geradas no Diagnóstico 05.

ESC e PROVA serão testadas separadamente. Cada cenário será executado em um subprocesso independente, para evitar que a memória de um teste contamine o seguinte.

**Nenhum arquivo original será modificado.**


## 1. Configuração do projeto

O notebook procura a raiz do projeto e o módulo `utils.features_names.py` (ou `src/utils/features_names.py`). A partir dele, importa `CONSUMO_DIR`, `DATA_DIR`, `_criar_mapa_nomes` e `_preparar_tabela`.

Se a estrutura local tiver outro caminho, ajuste `PROJECT_ROOT` na primeira célula.


In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import psutil
import gc
import json
import subprocess
import shutil
from IPython.display import display

# Localização do projeto: mesma lógica do Diagnóstico 05
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "dash_utils" / "config.py").exists():
    candidatos = [
        Path.cwd().parent,
        Path.cwd() / "app_render",
        Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render",
    ]

    PROJECT_ROOT = next(
        (
            p.resolve()
            for p in candidatos
            if (p / "dash_utils" / "config.py").exists()
        ),
        None
    )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não encontrei dash_utils/config.py. Ajuste PROJECT_ROOT."
    )

# Importação dos módulos do dashboard
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config = importlib.import_module("dash_utils.config")
features_names = importlib.import_module("dash_utils.features_names")

CONSUMO_DIR = Path(config.CONSUMO_DIR)
DATA_DIR = Path(config.DATA_DIR)

# Configuração da UF
UF = "PA"
UF_DIR = CONSUMO_DIR / f"uf={UF}"

SHAPE_PATH = (
    DATA_DIR
    / "shapefiles"
    / f"{UF}_Municipios_2025"
    / f"{UF}_Municipios_2025.shp"
)

# Arquivos originais
ARQUIVOS_ORIGINAIS = {
    "painel": UF_DIR / "painel_municipal.parquet",
    "faixa": UF_DIR / "faixa_etaria_municipal.parquet",
    "grupo": UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

# Variantes particionadas do Diagnóstico 05
DIAG05_DIR = PROJECT_ROOT / "_diagnostico05_parquet"

ARQUIVOS_PARTICIONADOS = {
    "grupo": DIAG05_DIR / "grupo" / "particionado_referencia",
    "dsp": DIAG05_DIR / "dsp" / "particionado_referencia",
}

print("Raiz do projeto:", PROJECT_ROOT)
print("Módulo:", features_names.__file__)
print("CONSUMO_DIR:", CONSUMO_DIR)
print("DATA_DIR:", DATA_DIR)
print("Shapefile:", SHAPE_PATH)
print("PyArrow:", pa.__version__)
print("Memory Pool:", pa.default_memory_pool().backend_name)

Raiz do projeto: /home/akel/PycharmProjects/Projeto_ENEM/app_render
Módulo: /home/akel/PycharmProjects/Projeto_ENEM/app_render/dash_utils/features_names.py
CONSUMO_DIR: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo
DATA_DIR: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data
Shapefile: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/shapefiles/PA_Municipios_2025/PA_Municipios_2025.shp
PyArrow: 26.0.1
Memory Pool: mimalloc


## 2. Conferência dos arquivos e colunas

As projeções abaixo reproduzem as listas definidas em `read_allfiles2()`. Antes de executar as medições, verificamos se todos os arquivos necessários existem e se as variantes particionadas do Diagnóstico 05 estão disponíveis.


In [2]:
COLUNAS = {
    "painel": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "n_participantes", "n_masculino", "n_feminino",
        "n_faixa_etaria_informada", "n_escola_federal",
        "n_escola_estadual", "n_escola_municipal", "n_escola_privada",
        "n_escola_nao_informada", "media_redacao", "n_validos_redacao",
        "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn",
        "media_ch", "n_validos_ch", "media_lc", "n_validos_lc",
        "media_mt", "n_validos_mt",
    ],
    "faixa": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA",
        "n_participantes",
    ],
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL", "tipo_grupo", "grupo",
        "disciplina", "media", "n_validos", "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL", "sexo", "faixa_etaria",
        "disciplina", "n_participantes", "n_validos", "media", "std",
    ],
}

for nome, caminho in ARQUIVOS_ORIGINAIS.items():
    if not caminho.exists():
        raise FileNotFoundError(caminho)
    pf = pq.ParquetFile(caminho)
    print(
        f"{nome:7s} | {pf.metadata.num_rows:>9,} linhas | "
        f"{caminho.stat().st_size / 1024**2:8.2f} MB | "
        f"{pf.metadata.num_row_groups} Row Group(s)"
    )

if not SHAPE_PATH.exists():
    raise FileNotFoundError(f"Shapefile não encontrado: {SHAPE_PATH}")

for nome, caminho in ARQUIVOS_PARTICIONADOS.items():
    if not caminho.exists():
        raise FileNotFoundError(
            f"Variante particionada não encontrada: {caminho}. "
            "Execute a geração correspondente no Diagnóstico 05."
        )
    dataset = ds.dataset(str(caminho), format="parquet", partitioning="hive")
    print(f"{nome:7s} | {len(list(dataset.get_fragments()))} fragmentos particionados")


painel  |     6,464 linhas |     1.19 MB | 1 Row Group(s)
faixa   |    63,836 linhas |     0.14 MB | 1 Row Group(s)
grupo   |   337,214 linhas |     8.57 MB | 1 Row Group(s)
dsp     |   451,205 linhas |    10.10 MB | 1 Row Group(s)
grupo   | 3 fragmentos particionados
dsp     | 3 fragmentos particionados


## 3. Regras de leitura dos cenários

A leitura das bases originais reproduz `pd.read_parquet()` com as mesmas colunas e filtros do `read_allfiles2()`.

No cenário particionado, somente `grupo` e `dsp` usam `pyarrow.dataset.Scanner`, com filtro aplicado sobre a coluna de partição. `painel` e `faixa` permanecem com a leitura original.

Depois da leitura, as duas estratégias usam exatamente as funções do projeto:

- `features_names._criar_mapa_nomes(shape)`
- `features_names._preparar_tabela(df, mapa_nomes, referencia)`

A função `_preparar_tabela()` substitui `RESIDENCIA` por `ESC`, aplica o filtro final, harmoniza o município e remove registros sem correspondência na malha.


In [3]:
WORKER = r'''
import os, sys, json, gc, time, logging
from pathlib import Path
import pandas as pd
import geopandas as gpd
import pyarrow.dataset as ds
import psutil

p = json.loads(sys.argv[1])
sys.path.insert(0, p["project_root"])

from dash_utils import features_names as fn

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    stream=sys.stdout,
)
proc = psutil.Process()

COLUNAS = p["columns"]
ARQ = p["original_files"]
PART = p["partitioned_files"]
SHAPE_PATH = p["shape_path"]
REF = p["referencia"]
CENARIO = p["cenario"]

def memoria():
    m = proc.memory_full_info()
    return {"rss_mb": m.rss / 1024**2, "uss_mb": m.uss / 1024**2}

def registrar(nome, inicio, antes):
    depois = memoria()
    print(
        f"ETAPA | {nome:34s} | "
        f"RSS={depois['rss_mb']:.1f} MB "
        f"(delta {depois['rss_mb']-antes['rss_mb']:+.1f}) | "
        f"USS={depois['uss_mb']:.1f} MB "
        f"(delta {depois['uss_mb']-antes['uss_mb']:+.1f}) | "
        f"tempo={time.perf_counter()-inicio:.2f} s",
        flush=True,
    )
    return depois

def ler_particionado(caminho, colunas, valores):
    dataset = ds.dataset(
        str(caminho),
        format="parquet",
        partitioning="hive",
    )
    scanner = dataset.scanner(
        columns=colunas,
        filter=ds.field("TIPO_REFERENCIA_TERRITORIAL").isin(valores),
        use_threads=True,
    )
    return scanner.to_table().to_pandas()

baseline = memoria()
print("BASELINE", json.dumps(baseline), flush=True)
t_total = time.perf_counter()

valores = ["ESC", "RESIDENCIA"] if REF == "ESC" else ["PROVA"]
filtros = [("TIPO_REFERENCIA_TERRITORIAL", "in", valores)]

dados = {}

for nome in ["painel", "faixa", "grupo", "dsp"]:
    antes = memoria()
    inicio = time.perf_counter()

    if CENARIO == "particionado" and nome in ("grupo", "dsp"):
        dados[nome] = ler_particionado(PART[nome], COLUNAS[nome], valores)
    else:
        dados[nome] = pd.read_parquet(
            ARQ[nome],
            engine="pyarrow",
            columns=COLUNAS[nome],
            filters=filtros,
        )

    registrar("leitura " + nome, inicio, antes)

antes = memoria()
inicio = time.perf_counter()
shape = gpd.read_file(SHAPE_PATH)
registrar("leitura shapefile", inicio, antes)

antes = memoria()
inicio = time.perf_counter()
mapa_nomes = fn._criar_mapa_nomes(shape)
registrar("criar mapa de nomes", inicio, antes)

for nome in ["painel", "faixa", "grupo", "dsp"]:
    antes = memoria()
    inicio = time.perf_counter()
    dados[nome] = fn._preparar_tabela(dados[nome], mapa_nomes, REF)
    registrar("preparar " + nome, inicio, antes)

print(
    "LINHAS FINAIS",
    json.dumps({k: len(v) for k, v in dados.items()}),
    flush=True,
)

# Assinaturas calculadas no próprio subprocesso para validar equivalência
# sem precisar manter os cenários simultaneamente na memória.
assinaturas = {}
for nome, df in dados.items():
    h = pd.util.hash_pandas_object(df, index=False).to_numpy(dtype="uint64")
    assinaturas[nome] = {
        "linhas": len(df),
        "hash_soma_uint64": int(h.sum(dtype="uint64")),
        "hash_xor_uint64": int(__import__("numpy").bitwise_xor.reduce(h, initial=0)),
        "colunas": list(df.columns),
    }

antes = memoria()
print("MEMORIA_ANTES_LIBERACAO", json.dumps(antes), flush=True)
del dados, shape, mapa_nomes
gc.collect()
depois = memoria()
print("MEMORIA_APOS_GC", json.dumps(depois), flush=True)

resultado = {
    "cenario": CENARIO,
    "referencia": REF,
    "baseline": baseline,
    "antes_liberacao": antes,
    "apos_gc": depois,
    "tempo_total_s": time.perf_counter() - t_total,
    "assinaturas": assinaturas,
}
print("RESULTADO_JSON", json.dumps(resultado), flush=True)
'''

print("Worker definido.")


Worker definido.


## 4. Função de execução isolada

Cada combinação será executada em um processo Python novo. Isso torna os deltas de RSS/USS mais comparáveis e evita que a memória retida pelo alocador de uma execução influencie a seguinte.

O subprocesso imprime as medições de cada etapa e devolve um resumo estruturado ao notebook.


In [4]:
PAYLOAD_BASE = {
    "project_root": str(PROJECT_ROOT),
    "columns": COLUNAS,
    "original_files": {
        k: str(v) for k, v in ARQUIVOS_ORIGINAIS.items()
    },
    "partitioned_files": {
        k: str(v) for k, v in ARQUIVOS_PARTICIONADOS.items()
    },
    "shape_path": str(SHAPE_PATH),
}

def executar_cenario(cenario, referencia):
    payload = {
        **PAYLOAD_BASE,
        "cenario": cenario,
        "referencia": referencia,
    }
    proc = subprocess.run(
        [sys.executable, "-c", WORKER, json.dumps(payload)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(
            f"Falha no cenário={cenario}, referência={referencia}\n"
            f"STDERR:\n{proc.stderr}"
        )

    linhas = [
        linha[len("RESULTADO_JSON "):]
        for linha in proc.stdout.splitlines()
        if linha.startswith("RESULTADO_JSON ")
    ]
    if not linhas:
        raise RuntimeError("O subprocesso não retornou RESULTADO_JSON.")
    return json.loads(linhas[-1])


## 5. Execução dos cenários

Execute os quatro testes. Cada cenário usa a mesma UF, referência, projeções, shapefile e preparação territorial. A única diferença é a origem física das bases `grupo` e `dsp`.


In [5]:
resultados_integrados = []

for referencia in ["ESC", "PROVA"]:
    for cenario in ["original", "particionado"]:
        print("\n" + "=" * 90)
        print(f"EXECUÇÃO: {cenario.upper()} | REFERÊNCIA: {referencia}")
        print("=" * 90)
        resultado = executar_cenario(cenario, referencia)
        resultados_integrados.append(resultado)

print("\nExecuções concluídas:", len(resultados_integrados))



EXECUÇÃO: ORIGINAL | REFERÊNCIA: ESC
BASELINE {"rss_mb": 135.0, "uss_mb": 67.58984375}
ETAPA | leitura painel                     | RSS=172.2 MB (delta +37.2) | USS=97.6 MB (delta +30.0) | tempo=0.01 s
ETAPA | leitura faixa                      | RSS=200.6 MB (delta +28.4) | USS=126.0 MB (delta +28.4) | tempo=0.01 s
ETAPA | leitura grupo                      | RSS=340.8 MB (delta +140.2) | USS=266.2 MB (delta +140.2) | tempo=0.03 s
ETAPA | leitura dsp                        | RSS=393.4 MB (delta +52.6) | USS=318.8 MB (delta +52.6) | tempo=0.03 s
ETAPA | leitura shapefile                  | RSS=428.5 MB (delta +35.1) | USS=330.9 MB (delta +12.1) | tempo=0.04 s
ETAPA | criar mapa de nomes                | RSS=429.5 MB (delta +1.0) | USS=330.9 MB (delta +0.0) | tempo=0.00 s
ETAPA | preparar painel                    | RSS=430.1 MB (delta +0.6) | USS=331.2 MB (delta +0.3) | tempo=0.01 s
ETAPA | preparar faixa                     | RSS=432.5 MB (delta +2.3) | USS=333.5 MB (delta +2.3) | te

## 6. Resumo de memória e tempo

O resumo separa:

- memória inicial do processo;
- memória depois de carregar e preparar todas as bases;
- memória depois de excluir os DataFrames e executar `gc.collect()`;
- tempo total do fluxo.

A métrica principal para comparação é o USS depois da preparação territorial, antes da liberação dos DataFrames. O USS após `gc.collect()` ajuda a observar quanto da memória permanece retida pelo processo.


In [6]:
def resumir_integrado(r):
    b = r["baseline"]
    a = r["antes_liberacao"]
    g = r["apos_gc"]
    return {
        "cenario": r["cenario"],
        "referencia": r["referencia"],
        "uss_inicial_mb": b["uss_mb"],
        "uss_carregado_mb": a["uss_mb"],
        "delta_uss_mb": a["uss_mb"] - b["uss_mb"],
        "rss_carregado_mb": a["rss_mb"],
        "delta_rss_mb": a["rss_mb"] - b["rss_mb"],
        "uss_apos_gc_mb": g["uss_mb"],
        "uss_retido_delta_mb": g["uss_mb"] - b["uss_mb"],
        "tempo_total_s": r["tempo_total_s"],
    }

df_integrado = pd.DataFrame([resumir_integrado(r) for r in resultados_integrados])
display(df_integrado.sort_values(["referencia", "cenario"]))


,cenario,referencia,uss_inicial_mb,uss_carregado_mb,delta_uss_mb,rss_carregado_mb,delta_rss_mb,uss_apos_gc_mb,uss_retido_delta_mb,tempo_total_s
0,original,ESC,67.589844,360.250000,292.660156,459.925781,324.925781,353.136719,285.546875,0.428217
1,particionado,ESC,67.535156,405.324219,337.789062,504.824219,369.863281,398.210938,330.675781,0.343962
2,original,PROVA,67.597656,335.648438,268.050781,434.835938,300.085938,334.640625,267.042969,0.276231
3,particionado,PROVA,67.621094,290.761719,223.140625,390.003906,255.097656,289.753906,222.132812,0.243551


## 7. Comparação direta: particionado versus original

A tabela calcula a variação percentual do USS carregado, RSS carregado e tempo total. Valores negativos representam redução no cenário particionado.

Como a execução de cada cenário ocorre em processo independente, compare principalmente as diferenças entre os cenários da mesma referência. Pequenas variações de baseline podem ocorrer entre processos.


In [7]:
orig = (
    df_integrado[df_integrado["cenario"] == "original"]
    .drop(columns="cenario")
    .set_index("referencia")
    .add_suffix("_original")
)
part = (
    df_integrado[df_integrado["cenario"] == "particionado"]
    .drop(columns="cenario")
    .set_index("referencia")
    .add_suffix("_particionado")
)

comparacao = orig.join(part, how="inner").reset_index()

for metrica in ["uss_carregado_mb", "delta_uss_mb", "rss_carregado_mb", "tempo_total_s"]:
    comparacao[f"variacao_{metrica}_pct"] = (
        (
            comparacao[f"{metrica}_particionado"]
            / comparacao[f"{metrica}_original"]
        ) - 1
    ) * 100

display(comparacao)


,referencia,uss_inicial_mb_original,uss_carregado_mb_original,delta_uss_mb_original,rss_carregado_mb_original,delta_rss_mb_original,uss_apos_gc_mb_original,uss_retido_delta_mb_original,tempo_total_s_original,uss_inicial_mb_particionado,...,delta_uss_mb_particionado,rss_carregado_mb_particionado,delta_rss_mb_particionado,uss_apos_gc_mb_particionado,uss_retido_delta_mb_particionado,tempo_total_s_particionado,variacao_uss_carregado_mb_pct,variacao_delta_uss_mb_pct,variacao_rss_carregado_mb_pct,variacao_tempo_total_s_pct
0,ESC,67.589844,360.250000,292.660156,459.925781,324.925781,353.136719,285.546875,0.428217,67.535156,...,337.789062,504.824219,369.863281,398.210938,330.675781,0.343962,12.511927,15.420243,9.762105,-19.675624
1,PROVA,67.597656,335.648438,268.050781,434.835938,300.085938,334.640625,267.042969,0.276231,67.621094,...,223.140625,390.003906,255.097656,289.753906,222.132812,0.243551,-13.373135,-16.754346,-10.310103,-11.830646


## 8. Equivalência dos DataFrames preparados

As assinaturas foram calculadas depois da preparação territorial, incluindo a coluna `NO_MUNICIPIO_TERRITORIO`.

Para cada referência, comparamos original e particionado por:
- número de linhas;
- soma dos hashes;
- XOR dos hashes;
- lista e ordem das colunas.

As assinaturas são calculadas em cada subprocesso, sem manter os DataFrames dos dois cenários simultaneamente na memória.


In [8]:
linhas_validacao = []

for r in resultados_integrados:
    for base, assinatura in r["assinaturas"].items():
        linhas_validacao.append({
            "cenario": r["cenario"],
            "referencia": r["referencia"],
            "base": base,
            **assinatura,
        })

df_assinaturas = pd.DataFrame(linhas_validacao)

comparacoes_validacao = []
for referencia in ["ESC", "PROVA"]:
    for base in ["painel", "faixa", "grupo", "dsp"]:
        sub = df_assinaturas[
            (df_assinaturas["referencia"] == referencia)
            & (df_assinaturas["base"] == base)
        ].set_index("cenario")

        if {"original", "particionado"}.issubset(sub.index):
            a = sub.loc["original"]
            b = sub.loc["particionado"]
            comparacoes_validacao.append({
                "referencia": referencia,
                "base": base,
                "linhas_iguais": a["linhas"] == b["linhas"],
                "hash_soma_igual": a["hash_soma_uint64"] == b["hash_soma_uint64"],
                "hash_xor_igual": a["hash_xor_uint64"] == b["hash_xor_uint64"],
                "colunas_iguais": a["colunas"] == b["colunas"],
            })

df_validacao_integrada = pd.DataFrame(comparacoes_validacao)
display(df_validacao_integrada)


,referencia,base,linhas_iguais,hash_soma_igual,hash_xor_igual,colunas_iguais
0,ESC,painel,True,True,True,True
1,ESC,faixa,True,True,True,True
2,ESC,grupo,True,True,True,True
3,ESC,dsp,True,True,True,True
4,PROVA,painel,True,True,True,True
5,PROVA,faixa,True,True,True,True
6,PROVA,grupo,True,True,True,True
7,PROVA,dsp,True,True,True,True


## 9. Comparação das quantidades de linhas

Esta tabela facilita a conferência de que o filtro e a preparação territorial produziram as mesmas quantidades de registros nos dois cenários.


In [9]:
df_linhas = df_assinaturas[
    ["cenario", "referencia", "base", "linhas"]
].pivot_table(
    index=["referencia", "base"],
    columns="cenario",
    values="linhas",
    aggfunc="first",
).reset_index()

if {"original", "particionado"}.issubset(df_linhas.columns):
    df_linhas["diferenca_linhas"] = df_linhas["particionado"] - df_linhas["original"]

display(df_linhas)


cenario,referencia,base,original,particionado,diferenca_linhas
0,ESC,dsp,250436,250436,0
1,ESC,faixa,38335,38335,0
2,ESC,grupo,186449,186449,0
3,ESC,painel,3625,3625,0
4,PROVA,dsp,187444,187444,0
5,PROVA,faixa,23177,23177,0
6,PROVA,grupo,131814,131814,0
7,PROVA,painel,1339,1339,0


## 10. Como interpretar os resultados

### Caso 1 — O ganho isolado permanece no carregamento integrado

Se USS e tempo total forem menores no cenário particionado, há evidência para considerar a adoção da organização física no pipeline de geração das bases `grupo` e `dsp`.

### Caso 2 — O ganho isolado diminui ou desaparece

O custo das demais bases, do shapefile ou da preparação territorial pode dominar o consumo total. Nesse caso, o particionamento ainda pode ser útil para leitura isolada, mas não resolve o pico integrado.

### Caso 3 — USS melhora, mas RSS ou memória após GC não

Isso pode indicar diferença entre memória privada e memória residente compartilhada/retida. Não devemos tratar RSS, USS e memória do Memory Pool como métricas equivalentes.

### Caso 4 — Dados não equivalentes

Se alguma assinatura divergir, não considerar a variante particionada pronta para produção. Primeiro investigar diferenças de tipos, valores nulos, colunas ou ordenação.

## Critério para avançar

Só recomendar a alteração do pipeline se:
1. os DataFrames preparados forem equivalentes;
2. o ganho de memória for consistente nas referências testadas;
3. o tempo não apresentar regressão relevante;
4. o ganho integrado for suficiente para justificar a mudança operacional.

O teste não presume que a memória necessariamente cairá: o resultado também pode mostrar que o custo dominante é inerente ao volume de dados ou a outras etapas do carregamento.


## 11. Exportar resultados

Opcionalmente, salve as tabelas-resumo para documentar a comparação. Os CSVs serão gravados na pasta `_diagnostico06_resultados`, sem alterar os dados do projeto.


In [10]:
PASTA_RESULTADOS = PROJECT_ROOT / "_diagnostico06_resultados"
PASTA_RESULTADOS.mkdir(exist_ok=True)

df_integrado.to_csv(PASTA_RESULTADOS / "resumo_memoria_tempo.csv", index=False)
comparacao.to_csv(PASTA_RESULTADOS / "comparacao_original_particionado.csv", index=False)
df_validacao_integrada.to_csv(PASTA_RESULTADOS / "validacao_equivalencia.csv", index=False)
df_linhas.to_csv(PASTA_RESULTADOS / "comparacao_linhas.csv", index=False)

print("Resultados salvos em:", PASTA_RESULTADOS)


Resultados salvos em: /home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnostico06_resultados
